In [ ]:
# Smoke test: validate_email is a single-file module (not a package),
# so there is no __path__ to walk. Import + sanity-check the public surface.
import validate_email as ve_module
from validate_email import validate_email

print(f"validate_email module file: {ve_module.__file__}")
print(f"validate_email function: {validate_email}")

# Public names we expect on the module.
expected = ['validate_email', 'VALID_ADDRESS_REGEXP', 'get_mx_ip', 'ServerError']
missing = [n for n in expected if not hasattr(ve_module, n)]
if missing:
    print(f"\nMISSING attributes: {missing}")
else:
    print(f"\nAll expected public names present: {expected}")

In [ ]:
# Example 1: basic syntactic validation (default mode, no MX / SMTP).
from validate_email import validate_email

valid_addresses = [
    'user@example.com',
    'first.last@sub.example.org',
    'user+tag@example.io',
    'a@b.co',
]
for addr in valid_addresses:
    ok = validate_email(addr)
    print(f"{addr!r:40s} -> {ok}")
    assert ok is True, f"expected {addr} to validate"

In [ ]:
# Example 2: rejection of malformed addresses.
from validate_email import validate_email

invalid_addresses = [
    'plainaddress',
    '@missing-local.org',
    'user@',
    'user@@example.com',
    'user name@example.com',  # space in local-part
    '',
]
for addr in invalid_addresses:
    ok = validate_email(addr)
    print(f"{addr!r:40s} -> {ok}")
    assert not ok, f"expected {addr!r} to be rejected"

In [ ]:
# Example 3: the RFC 2822 addr-spec regex is exposed as VALID_ADDRESS_REGEXP.
import re

from validate_email import VALID_ADDRESS_REGEXP

# The regex object should be a compiled pattern (or at least a string usable
# with re.match — depending on the library version it can be either).
if hasattr(VALID_ADDRESS_REGEXP, 'match'):
    pattern = VALID_ADDRESS_REGEXP
else:
    pattern = re.compile(VALID_ADDRESS_REGEXP)

m = pattern.match('user@example.com')
print(f"match for 'user@example.com': {m}")
assert m is not None

m2 = pattern.match('not-an-email')
print(f"match for 'not-an-email':    {m2}")
assert m2 is None

In [ ]:
# Example 4: ServerError is exported and is an Exception subclass.
from validate_email import ServerError

assert isinstance(ServerError, type), "ServerError should be a class"
assert issubclass(ServerError, Exception), "ServerError should subclass Exception"

# It should be raise-able and catchable like a normal exception.
try:
    raise ServerError('synthetic test')
except ServerError as e:
    print(f"caught ServerError: {e}")
    caught = True

assert caught

In [ ]:
# Example 5: end-to-end syntactic-validation summary (no network — check_mx /
# verify are deliberately False so the test stays self-contained).
from validate_email import validate_email

cases = {
    'alice@example.com':       True,
    'bob.smith+filter@x.io':   True,
    'no-at-sign':              False,
    'two@@ats.com':            False,
    'trailing-dot@.com':       False,
}

results = {addr: bool(validate_email(addr, check_mx=False, verify=False))
           for addr in cases}

for addr, expected in cases.items():
    got = results[addr]
    status = 'OK' if got == expected else 'FAIL'
    print(f"[{status}] {addr!r:30s} expected={expected} got={got}")

assert results == cases, f"mismatch: {results} vs {cases}"